## 1.8 Visualization for Data Cleaning

In [ ]:
"""
HMDA Preprocessing Validation — 5 Separate Figures  (text-overlap-fixed)
=========================================================================
Outputs:
    fig1_imputation.png
    fig2_winsorization.png
    fig3_binning.png
    fig4_feature_funnel.png
    fig5_geographic.png

Run:
    python preprocessing_eda_fixed.py
"""

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.patheffects as path_effects
from scipy import stats
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ── 0. Load & prep ────────────────────────────────────────────────────────────
CSV_PATH = "HMDA_CA_TX_2019_2024_Master.csv"

df_raw = pd.read_csv(CSV_PATH, low_memory=False)

NUMERIC = [
    "loan_amount", "income", "property_value", "interest_rate", "rate_spread",
    "total_loan_costs", "origination_charges", "discount_points", "lender_credits",
    "loan_to_value_ratio", "tract_population", "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income", "tract_to_msa_income_percentage",
    "tract_owner_occupied_units", "tract_one_to_four_family_homes",
    "tract_median_age_of_housing_units",
]
for c in NUMERIC:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")
df_raw["target_y"]     = pd.to_numeric(df_raw["target_y"],     errors="coerce")
df_raw["loan_purpose"] = pd.to_numeric(df_raw["loan_purpose"], errors="coerce")
df_raw["aus-1"]        = pd.to_numeric(df_raw["aus-1"],        errors="coerce")

X = df_raw.drop(columns=["target_y"])
y = df_raw["target_y"]
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
df_train = X_train.copy()
df_train["target_y"] = y_train.values

# ── Palette ───────────────────────────────────────────────────────────────────
BLUE   = "#2563EB"
CORAL  = "#F97316"
TEAL   = "#0D9488"
VIOLET = "#7C3AED"
AMBER  = "#D97706"
GRAY   = "#9CA3AF"
ROSE   = "#E11D48"
SLATE  = "#334155"

BG      = "#F8FAFC"
PANEL   = "#FFFFFF"
BORDER  = "#E2E8F0"
TEXT_HI = "#0F172A"
TEXT_LO = "#64748B"

matplotlib.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.edgecolor":     BORDER,
    "axes.facecolor":     PANEL,
    "figure.facecolor":   BG,
    "grid.color":         BORDER,
    "grid.linewidth":     0.6,
    "xtick.color":        TEXT_LO,
    "ytick.color":        TEXT_LO,
    "text.color":         TEXT_HI,
})

# ── Shared helpers ────────────────────────────────────────────────────────────
def fig_header(fig, title, subtitle=""):
    """Two-line header with enough top margin so nothing overlaps the plot."""
    fig.suptitle(title, fontsize=14, fontweight="bold", color=TEXT_HI, y=0.99)
    if subtitle:
        fig.text(0.5, 0.955, subtitle, ha="center", fontsize=9.5,
                 color=TEXT_LO, wrap=True)

def ax_header(ax, title, note=""):
    """Bold left-aligned axis title + small italic note on a separate line."""
    ax.set_title(title, fontsize=11.5, fontweight="bold", color=TEXT_HI,
                 loc="left", pad=6)
    if note:
        # Place note below the title, inside the axes top area
        ax.text(0.0, 1.055, note, transform=ax.transAxes,
                fontsize=8, color=TEXT_LO, va="bottom", ha="left",
                style="italic")

def callout_box(ax, txt, x=0.97, y=0.97, color=TEAL, bg="#F0FDF4", ha="right", va="top"):
    ax.text(x, y, txt, transform=ax.transAxes, ha=ha, va=va,
            fontsize=8.5, color=color,
            bbox=dict(boxstyle="round,pad=0.4", fc=bg,
                      ec=color, alpha=0.92, linewidth=0.8))

### Figure 1 — Imputation Validation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIG 1 — Imputation Validation  (Income Median vs DTI Mean)
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering fig1_imputation.png ...")

fig1, axes1 = plt.subplots(1, 2, figsize=(15, 6),
                            facecolor=BG,
                            gridspec_kw={"wspace": 0.30})

fig1.subplots_adjust(top=0.84, bottom=0.12, left=0.07, right=0.97)
fig_header(fig1, "① Imputation Validation — KDE Before vs After")

# ── Left: Income ──────────────────────────────────────
ax = axes1[0]
income_all = df_train["income"]
income_obs = income_all.dropna()
income_med = income_obs.median()
income_imp = income_all.fillna(income_med)

cap = income_obs.quantile(0.99)
obs_c = income_obs[income_obs <= cap]
imp_c = income_imp[income_imp <= cap]

kde_b = stats.gaussian_kde(obs_c, bw_method=0.15)
kde_a = stats.gaussian_kde(imp_c, bw_method=0.15)
xg    = np.linspace(0, cap, 500)

ax.fill_between(xg, kde_b(xg), alpha=0.15, color=BLUE)
ax.fill_between(xg, kde_a(xg), alpha=0.15, color=CORAL)
ax.plot(xg, kde_b(xg), lw=2.0, color=BLUE, label=f"Before (n={len(obs_c):,})")
ax.plot(xg, kde_a(xg), lw=2.0, color=CORAL, ls="--", label=f"After (n={len(imp_c):,})")
ax.axvline(income_med, color=AMBER, lw=1.6, ls=":", label=f"Median Fill = {income_med:.0f}k")

xz = np.linspace(income_med - (cap * 0.025), income_med + (cap * 0.025), 60)
ax.fill_between(xz, kde_a(xz), alpha=0.4, color=AMBER)

ax.set_xlabel("Income (×$1 k)", fontsize=10, color=TEXT_LO)
ax.set_ylabel("Density", fontsize=10, color=TEXT_LO)
ax.legend(fontsize=8.5, framealpha=0, loc="upper right", bbox_to_anchor=(1.0, 0.62))
ax_header(ax, "Income", f"Missing {income_all.isna().mean()*100:.1f}% · median strategy")

# ── Right: Debt-to-Income Ratio ───────────────
ax = axes1[1]

dti_all = df_cleaned["debt_to_income_ratio"]
dti_obs = dti_all.dropna()
dti_mean = dti_obs.mean()
dti_imp = dti_all.fillna(dti_mean)

cap_dti = dti_obs.quantile(0.99)
obs_d   = dti_obs[dti_obs <= cap_dti]
imp_d   = dti_imp[dti_imp <= cap_dti]

kde_d_b = stats.gaussian_kde(obs_d, bw_method=0.12)
kde_d_a = stats.gaussian_kde(imp_d, bw_method=0.12)
xg_d    = np.linspace(dti_obs.min(), cap_dti, 500)

ax.fill_between(xg_d, kde_d_b(xg_d), alpha=0.15, color=VIOLET)
ax.fill_between(xg_d, kde_d_a(xg_d), alpha=0.15, color=TEAL)
ax.plot(xg_d, kde_d_b(xg_d), lw=2.0, color=VIOLET, label=f"Before (n={len(obs_d):,})")
ax.plot(xg_d, kde_d_a(xg_d), lw=2.0, color=TEAL, ls="--", label=f"After (n={len(imp_d):,})")

ax.axvline(dti_mean, color=AMBER, lw=1.6, ls=":", label=f"Mean Fill = {dti_mean:.1f}%")

xz_d = np.linspace(dti_mean - 1.5, dti_mean + 1.5, 60)
ax.fill_between(xz_d, kde_d_a(xz_d), alpha=0.4, color=AMBER)

ax.set_xlabel("Debt-to-Income Ratio (%)", fontsize=10, color=TEXT_LO)
ax.set_ylabel("Density", fontsize=10, color=TEXT_LO)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
ax.xaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
ax.legend(fontsize=8.5, framealpha=0, loc="upper right")

ax_header(ax, "Debt-to-Income Ratio", 
          f"Missing {dti_all.isna().mean()*100:.1f}% · mean fill = {dti_mean:.1f}%"
          f" · Replaced interest_rate")

fig1.savefig("fig1_imputation.png", dpi=300, bbox_inches="tight", facecolor=BG)
print("  → Saved fig1_imputation.png")
plt.close(fig1)

### Figure 2 — Winsorization

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIG 2 — Winsorization Check
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering fig2_winsorization.png ...")

fig2, axes2 = plt.subplots(1, 2, figsize=(15, 6.5),
                            facecolor=BG,
                            gridspec_kw={"wspace": 0.35})
fig2.subplots_adjust(top=0.84, bottom=0.16, left=0.09, right=0.93)
fig_header(fig2,
           "② Winsorization Check — Outlier Capping (1st / 99th Percentile)")

def winsor_ax(ax, series_raw, unit_fmt, c_raw, c_win, note_title, note_sub):
    filled = series_raw.fillna(series_raw.median())
    p1  = filled.quantile(0.01)
    p99 = filled.quantile(0.99)
    winsorized = filled.clip(p1, p99)

    rng = np.random.default_rng(42)
    idx = rng.choice(len(filled), min(6000, len(filled)), replace=False)
    raw_s = filled.iloc[idx]
    win_s = winsorized.iloc[idx]

    bp_kw = dict(patch_artist=True, widths=0.42,
                 medianprops=dict(color="white", lw=2.5))

    ax.boxplot(raw_s, positions=[1], showfliers=True,
               flierprops=dict(marker=".", ms=1.8, alpha=0.12, color=GRAY),
               boxprops=dict(facecolor=c_raw, alpha=0.72, linewidth=0),
               whiskerprops=dict(color=c_raw, lw=1.4),
               capprops=dict(color=c_raw, lw=1.4), **bp_kw)

    ax.boxplot(win_s, positions=[2], showfliers=False,
               boxprops=dict(facecolor=c_win, alpha=0.85, linewidth=0),
               whiskerprops=dict(color=c_win, lw=1.4),
               capprops=dict(color=c_win, lw=1.4), **bp_kw)

    # p99 / p1 lines — label inside the axes, left-anchored to avoid right clip
    for yval, lbl in [(p99, f"p99 = {unit_fmt(p99)}"),
                      (p1,  f"p1  = {unit_fmt(p1)}")]:
        ax.axhline(yval, color=c_win, ls="--", lw=1.2, alpha=0.70)
        ax.text(0.52, yval, lbl, fontsize=8, va="center",
                color=c_win, transform=ax.get_yaxis_transform())

    pct_cap = ((filled < p1) | (filled > p99)).mean() * 100
    iqr_raw = raw_s.quantile(0.75) - raw_s.quantile(0.25)
    iqr_win = win_s.quantile(0.75) - win_s.quantile(0.25)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Raw", "Winsorized\n(1% – 99%)"], fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: unit_fmt(v)))
    ax.yaxis.grid(True, alpha=0.35); ax.set_axisbelow(True)
    ax.set_xlim(0.5, 2.8)

    # Stats line at bottom — use fig.text coords to avoid being clipped
    ax.set_xlabel(
        f"Capped: {pct_cap:.1f}% of values   ·   "
        f"IQR: {unit_fmt(iqr_raw)} → {unit_fmt(iqr_win)}",
        fontsize=8.5, color=TEXT_LO, labelpad=10)

    handles = [mpatches.Patch(color=c_raw, label="Raw (after median fill)"),
               mpatches.Patch(color=c_win, label="After winsorization")]
    ax.legend(handles=handles, fontsize=9, framealpha=0, loc="upper right")

    ax_header(ax, note_title, note_sub)

winsor_ax(axes2[0], df_train["income"],
          lambda v: f"${v:.0f}k", ROSE, TEAL,
          "Income",
          "Max raw = $855,121k  →  p99 cap = $855k  ·  Long right tail eliminated")
callout_box(axes2[0],
            "Prevents extreme outliers\nfrom biasing model weights",
            x=0.03, y=0.58, ha="left", va="top", color=TEAL, bg="#F0FDF4")

winsor_ax(axes2[1], df_train["loan_amount"],
          lambda v: f"${v/1e3:.0f}k", AMBER, BLUE,
          "Loan Amount",
          "Max raw = $196 M  →  p99 cap = $1,925k  ·  Jumbo-loan extremes contained")
callout_box(axes2[1],
            "Train-derived bounds applied\nto test set → no leakage",
            x=0.03, y=0.58, ha="left", va="top", color=BLUE, bg="#EFF6FF")

fig2.savefig("fig2_winsorization.png", dpi=300, bbox_inches="tight", facecolor=BG)
print("  → Saved fig2_winsorization.png")
plt.close(fig2)

### Figure 3 — Category Consolidation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIG 3 — Category Consolidation
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering fig3_binning.png ...")

fig3, axes3 = plt.subplots(1, 2, figsize=(16, 7),
                            facecolor=BG,
                            gridspec_kw={"wspace": 0.38})
fig3.subplots_adjust(top=0.84, bottom=0.10, left=0.14, right=0.97)
fig_header(fig3,
           "③ Category Consolidation — Approval Rate by Bin")

# ── Left: loan_purpose ────────────────────────────────────────────────────────
ax = axes3[0]
LP_MAP = {1: "Purchase", 31: "Refi — cash-out", 32: "Refi — no cash",
          2: "Home improvement", 4: "Other purpose", 5: "Construction"}
df_raw["_lp"] = df_raw["loan_purpose"].map(LP_MAP).fillna("Unknown")
lp = df_raw.groupby("_lp")["target_y"].agg(rate="mean", n="count").reset_index()
lp = lp.sort_values("rate")

STD_LP = {"Purchase", "Refi — cash-out", "Refi — no cash"}
bar_colors = [TEAL if r in STD_LP else ROSE for r in lp["_lp"]]

bars = ax.barh(lp["_lp"], lp["rate"] * 100, color=bar_colors,
               edgecolor="white", linewidth=0.5, height=0.55)

# Value labels: put them INSIDE bars (white) to avoid right-margin overflow
for bar, row in zip(bars, lp.itertuples()):
    w = bar.get_width()
    # Count label inside bar
    ax.text(w - 1.0, bar.get_y() + bar.get_height() / 2,
            f"{w:.1f}%", va="center", ha="right",
            fontsize=9.5, color="white", fontweight="bold")
    # n= label to the right, but only if there's room (xlim 0–100)
    ax.text(w + 0.8, bar.get_y() + bar.get_height() / 2,
            f"n={row.n:,}", va="center", ha="left",
            fontsize=8.5, color=TEXT_LO)

std_rates   = lp[lp["_lp"].isin(STD_LP)]["rate"]
hrisk_rates = lp[~lp["_lp"].isin(STD_LP)]["rate"]
gap = (std_rates.mean() - hrisk_rates.mean()) * 100

ax.axvline(std_rates.mean()   * 100, color=TEAL, ls=":", lw=1.4, alpha=0.7)
ax.axvline(hrisk_rates.mean() * 100, color=ROSE, ls=":", lw=1.4, alpha=0.7)

ax.set_xlim(0, 115)   # extra room for n= labels
ax.set_xlabel("Approval Rate (%)", fontsize=10, color=TEXT_LO)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.xaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

handles = [mpatches.Patch(color=TEAL, label="Standard Purpose"),
           mpatches.Patch(color=ROSE, label="High-Risk Purpose")]
ax.legend(handles=handles, fontsize=9, framealpha=0, loc="lower right")
callout_box(ax, f"Δ = {gap:.0f} pp between groups",
            x=0.97, y=0.50, ha="right", va="top", color=BLUE, bg="#EFF6FF")
ax_header(ax, "Loan Purpose — raw categories",
          "Standard: Purchase + Refi   ·   High-Risk: Improvement + Other")

# ── Right: aus-1 ──────────────────────────────────────────────────────────────
ax = axes3[1]
AUS_MAP = {1: "Fannie DU (1)", 2: "Freddie LPA (2)", 3: "Freddie other (3)",
           4: "VA TOTAL (4)", 5: "USDA GUS (5)", 6: "Other internal (6)",
           7: "Not applicable (7)", 1111: "Exempt (1111)"}
STD_AUS = {1, 2, 3, 4}
df_raw["_aus"] = df_raw["aus-1"].map(AUS_MAP).fillna("Unknown")
aus = df_raw.groupby("_aus")["target_y"].agg(rate="mean", n="count").reset_index()
aus = aus.sort_values("rate")

std_keys       = {AUS_MAP[k] for k in STD_AUS if k in AUS_MAP}
bar_colors_aus = [TEAL if r in std_keys else ROSE for r in aus["_aus"]]

bars = ax.barh(aus["_aus"], aus["rate"] * 100, color=bar_colors_aus,
               edgecolor="white", linewidth=0.5, height=0.55)

for bar, row in zip(bars, aus.itertuples()):
    w = bar.get_width()
    ax.text(w - 1.0, bar.get_y() + bar.get_height() / 2,
            f"{w:.1f}%", va="center", ha="right",
            fontsize=9.5, color="white", fontweight="bold")
    ax.text(w + 0.8, bar.get_y() + bar.get_height() / 2,
            f"n={row.n:,}", va="center", ha="left",
            fontsize=8.5, color=TEXT_LO)

std_aus_rates  = aus[aus["_aus"].isin(std_keys)]["rate"]
nstd_aus_rates = aus[~aus["_aus"].isin(std_keys)]["rate"]
gap_aus = (std_aus_rates.mean() - nstd_aus_rates.mean()) * 100

ax.axvline(std_aus_rates.mean()  * 100, color=TEAL, ls=":", lw=1.4, alpha=0.7)
ax.axvline(nstd_aus_rates.mean() * 100, color=ROSE, ls=":", lw=1.4, alpha=0.7)

# Inset: after grouping — positioned top-right, away from labels
ins = ax.inset_axes([0.55, 0.65, 0.43, 0.30])
df_raw["_aus_grp"] = df_raw["aus-1"].apply(
    lambda v: "Standard AUS" if v in STD_AUS else "Non-Standard /\nExempt")
grp = df_raw.groupby("_aus_grp")["target_y"].agg(rate="mean", n="count") \
            .reset_index().sort_values("rate")
ins_bars = ins.barh(grp["_aus_grp"], grp["rate"] * 100,
                    color=[ROSE, TEAL], height=0.45, edgecolor="white")
for b in ins_bars:
    ins.text(b.get_width() - 1.0, b.get_y() + b.get_height() / 2,
             f"{b.get_width():.1f}%", va="center", ha="right",
             fontsize=8, color="white", fontweight="bold")
ins.set_xlim(0, 105)
ins.set_title("After grouping", fontsize=8, color=TEXT_LO, loc="left", pad=3)
ins.tick_params(labelsize=7.5)
ins.spines["top"].set_visible(False); ins.spines["right"].set_visible(False)
ins.xaxis.grid(True, alpha=0.2); ins.set_axisbelow(True)
ins.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))

ax.set_xlim(0, 115)
ax.set_xlabel("Approval Rate (%)", fontsize=10, color=TEXT_LO)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.xaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

handles_aus = [mpatches.Patch(color=TEAL, label="Standard AUS (codes 1–4)"),
               mpatches.Patch(color=ROSE, label="Non-Standard / Exempt")]
ax.legend(handles=handles_aus, fontsize=9, framealpha=0, loc="lower right")
callout_box(ax, f"Δ = {gap_aus:.0f} pp between groups",
            x=0.97, y=0.50, ha="right", va="top", color=BLUE, bg="#EFF6FF")
ax_header(ax, "AUS-1 — raw categories",
          "Fannie/Freddie/VA cluster ~89%  ·  USDA / Other / Exempt drop to ~63%")

fig3.savefig("fig3_binning.png", dpi=300, bbox_inches="tight", facecolor=BG)
print("  → Saved fig3_binning.png")
plt.close(fig3)

### Figure 4 — Feature Funnel

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIG 4 — Feature Funnel
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering fig4_feature_funnel.png ...")

fig4, ax = plt.subplots(1, 1, figsize=(12, 8), facecolor=BG)

fig4.subplots_adjust(top=0.85, bottom=0.15, left=0.32, right=0.92)
fig_header(fig4, "④ Dimensionality Reduction — Feature Selection Funnel")

# ── Funnel Data (Verified Predictor Counts: 99 -> 51) ──────────────────────────
stages = [
    ("Raw predictors",                      99, SLATE),
    ("− Post-outcome leakage cols",         93, ROSE),
    ("− Columns >80% missing",              69, AMBER),
    ("− High-cardinality cat. (>8 unique)", 57, VIOLET),
    ("− Redundant / No-signal",             55, TEAL),
    ("✓ Final features (before OHE)",       51, BLUE),
]

labels = [s[0] for s in stages]
vals   = [s[1] for s in stages]
colors = [s[2] for s in stages]

bars = ax.barh(labels, vals, color=colors, edgecolor="white", 
               linewidth=0.8, height=0.65, alpha=0.9)
ax.invert_yaxis() # 让 Raw 处于最上方

for bar, val in zip(bars, vals):
    ax.text(bar.get_width() - 1.5,
            bar.get_y() + bar.get_height() / 2,
            f"{val}", va="center", ha="right",
            fontsize=12, fontweight="bold", color="white")

for i in range(1, len(stages)):
    delta = vals[i] - vals[i - 1]
    if delta != 0:
        y_mid = i - 0.5
        ax.text(vals[i] + 2, y_mid,
                f"{delta:+d} features", fontsize=10, 
                color=GRAY, fontweight="bold",
                va="center", ha="left", style='italic')

ax.set_xlim(0, 115) 
ax.set_xlabel("Number of predictive features", fontsize=11, color=TEXT_LO, labelpad=12)
ax.xaxis.grid(True, ls='--', alpha=0.3, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(axis="y", labelsize=11)

reduction_pct = (1 - (51/99)) * 100
callout_box(ax, 
            f"Overall Strategy:\n{reduction_pct:.1f}% reduction\n99 → 51 predictors",
            x=0.98, y=0.05, ha="right", va="bottom", color=BLUE, bg="#EFF6FF")

ax_header(ax, "Sequential Feature Filtering Pipeline",
          "Filtering logic applied strictly to training split to prevent leakage | Excludes target variable")

fig4.savefig("fig4_feature_funnel.png", dpi=300, bbox_inches="tight", facecolor=BG)
print("  → Saved fig4_feature_funnel.png")
plt.close(fig4)

### Figure 5 - Distribution between 2 States

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FIG 5 — Geographic Distribution
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering fig5_geographic.png ...")

ca = df_raw[df_raw["state_code"] == "CA"]
tx = df_raw[df_raw["state_code"] == "TX"]

fig5, axes5 = plt.subplots(1, 2, figsize=(15, 6),
                            facecolor=BG,
                            gridspec_kw={"wspace": 0.32})
fig5.subplots_adjust(top=0.84, bottom=0.14, left=0.07, right=0.97)
fig_header(fig5,
           "⑤ Geographic Distribution — California vs Texas")

# ── Left: count + approval rate ───────────────────────────────────────────────
ax = axes5[0]
counts    = [len(ca), len(tx)]
approvals = [ca["target_y"].mean() * 100, tx["target_y"].mean() * 100]
xlabels   = ["California (CA)", "Texas (TX)"]
x = np.arange(2)
w = 0.30

bars_cnt = ax.bar(x - w / 2, counts, w, color=[BLUE, TEAL],
                  alpha=0.85, edgecolor="white", zorder=3)
ax_r = ax.twinx()
bars_app = ax_r.bar(x + w / 2, approvals, w, color=[AMBER, ROSE],
                    alpha=0.85, edgecolor="white", zorder=3)

# Labels above each bar — give enough y-clearance
for bar, val in zip(bars_cnt, counts):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.03,
            f"{val:,}", ha="center", fontsize=10,
            fontweight="bold", color=TEXT_HI)

for bar, val in zip(bars_app, approvals):
    ax_r.text(bar.get_x() + bar.get_width() / 2,
              bar.get_height() + 1.5,
              f"{val:.1f}%", ha="center", fontsize=10,
              fontweight="bold", color=TEXT_HI)

ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=11)
ax.set_ylabel("Sample count", fontsize=10, color=TEXT_LO)
ax_r.set_ylabel("Approval rate (%)", fontsize=10, color=TEXT_LO)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e3:.0f}k"))
ax_r.set_ylim(0, 115)    # room for labels above bars
ax.set_ylim(0, max(counts) * 1.18)
ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

# Legend below the chart to avoid colliding with bar labels
handles5 = [mpatches.Patch(color=BLUE,  label="CA — count"),
            mpatches.Patch(color=TEAL,  label="TX — count"),
            mpatches.Patch(color=AMBER, label="CA — approval rate"),
            mpatches.Patch(color=ROSE,  label="TX — approval rate")]
ax.legend(handles=handles5, fontsize=8.5, framealpha=0,
          ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12))

ax_header(ax, "Sample size & approval rate",
          f"CA: {len(ca):,} ({len(ca)/len(df_raw)*100:.1f}%)  ·  "
          f"TX: {len(tx):,} ({len(tx)/len(df_raw)*100:.1f}%)  ·  "
          f"Δ approval = {abs(approvals[0]-approvals[1]):.1f} pp")

# ── Right: loan amount KDE ────────────────────────────────────────────────────
ax = axes5[1]
cap_la = df_raw["loan_amount"].quantile(0.99)
ca_la  = ca["loan_amount"].dropna(); ca_la = ca_la[ca_la <= cap_la]
tx_la  = tx["loan_amount"].dropna(); tx_la = tx_la[tx_la <= cap_la]

kde_ca = stats.gaussian_kde(ca_la, bw_method=0.14)
kde_tx = stats.gaussian_kde(tx_la, bw_method=0.14)
xg = np.linspace(0, cap_la, 500)

ax.fill_between(xg, kde_ca(xg), alpha=0.15, color=BLUE)
ax.fill_between(xg, kde_tx(xg), alpha=0.15, color=TEAL)
ax.plot(xg, kde_ca(xg), lw=2.2, color=BLUE,
        label=f"CA  median=${ca_la.median()/1e3:.0f}k  mean=${ca_la.mean()/1e3:.0f}k")
ax.plot(xg, kde_tx(xg), lw=2.2, color=TEAL,
        label=f"TX  median=${tx_la.median()/1e3:.0f}k  mean=${tx_la.mean()/1e3:.0f}k")
ax.axvline(ca_la.median(), color=BLUE, lw=1.3, ls=":", alpha=0.8)
ax.axvline(tx_la.median(), color=TEAL, lw=1.3, ls=":", alpha=0.8)

# Delta annotation — place well below the KDE peaks
y_ann = min(kde_ca(ca_la.median())[0], kde_tx(tx_la.median())[0]) * 0.22
diff  = ca_la.median() - tx_la.median()
ax.annotate("", xy=(ca_la.median(), y_ann),
            xytext=(tx_la.median(), y_ann),
            arrowprops=dict(arrowstyle="<->", color=GRAY, lw=1.3))
ax.text((ca_la.median() + tx_la.median()) / 2, y_ann * 1.25,
        f"Δ median = ${diff/1e3:.0f}k\nCA loans ~74% larger",
        ha="center", fontsize=8.5, color=TEXT_LO,
        bbox=dict(fc=BG, ec="none", alpha=0.8))

ax.set_xlabel("Loan Amount", fontsize=10, color=TEXT_LO)
ax.set_ylabel("Density", fontsize=10, color=TEXT_LO)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, _: f"${v/1e6:.1f}M" if v >= 1e6 else f"${v/1e3:.0f}k"))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.6f"))
ax.legend(fontsize=9, framealpha=0, loc="upper right")
ax.xaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)

callout_box(ax, "Distinct distributions:\nboth states add unique signal",
            x=0.03, y=0.97, ha="left", va="top", color=VIOLET, bg="#F5F3FF")
ax_header(ax, "Loan amount distribution — CA vs TX",
          "CA's higher property market shifts distribution rightward  ·  Capped at p99 for display")

fig5.savefig("fig5_geographic.png", dpi=300, bbox_inches="tight", facecolor=BG)
print("  → Saved fig5_geographic.png")
plt.close(fig5)

print("\nAll 5 figures saved successfully.")